# Stage 0 — CensoPersonas 2021: raw microdata → parquet

Downloads `CensoPersonas_2021.tab.zst` from S3, applies a commercial population filter,
selects modelling-relevant columns, and saves a pre-aggregation parquet.

Mirrors `0_transform_to_parquet.ipynb` from the France census project.

**Commercial filter (confirmed from codebook `dr_CensoPersonas_2021.xlsx`):**
- `TENEN_VIV IN ('2', '3')` — owner-occupied and private rented only; excludes social/free housing (`4`) and unknown (`9`)
- No mainland filter needed — file contains only Spanish residents (no overseas equivalent)
- No dwelling type filter needed — file already contains only household dwellings (no communal establishments)

**Columns dropped:** all `_MI` imputation flag columns, mobility/migration history columns,
nationality/birthplace detail columns, workplace/study location columns, and columns
derivable from others (SUP_OCU_VIV, NPLANTAS_BAJO_EDIF, FAM_HOG, NUC_HOG).

**S3 source:** `s3://hsf-group-ai-spain-hvac/ine-census-2021/raw/CensoPersonas_2021.tab.zst`  
**Local output:** `data/generated/ine_census_2021/censo_personas.parquet`  
**S3 output:** `s3://hsf-group-ai-spain-hvac/ine-census-2021/generated/censo_personas.parquet`

In [1]:
# CELL 1 — DOWNLOAD RAW MICRODATA FROM S3 (skip if already on disk)
import os, subprocess, pathlib

BUCKET   = "hsf-group-ai-spain-hvac"
S3_KEY   = "ine-census-2021/raw/CensoPersonas_2021.tab.zst"
S3_SRC   = f"s3://{BUCKET}/{S3_KEY}"
LOCAL_ZST = "data/input/ine_census_2021/CensoPersonas_2021.tab.zst"
PROFILE  = "AWSAdministratorAccess-268271485741"

pathlib.Path("data/input/ine_census_2021").mkdir(parents=True, exist_ok=True)

if os.path.exists(LOCAL_ZST):
    size_mb = os.path.getsize(LOCAL_ZST) / 1_048_576
    print(f"Already exists: {LOCAL_ZST}  ({size_mb:.0f} MB) — nothing to do.")
else:
    print(f"Downloading: {S3_SRC}")
    result = subprocess.run(
        ["aws", "s3", "cp", S3_SRC, LOCAL_ZST, "--profile", PROFILE],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        size_mb = os.path.getsize(LOCAL_ZST) / 1_048_576
        print(f"Downloaded: {LOCAL_ZST}  ({size_mb:.0f} MB)")
    else:
        print("Download failed — check AWS SSO login:")
        print(result.stderr)

Already exists: data/input/ine_census_2021/CensoPersonas_2021.tab.zst  (129 MB) — nothing to do.


In [2]:
# CELL 2 — VERIFY: sample 1,000 rows and check filter column distributions
import zstd, io, pandas as pd

LOCAL_ZST = "data/input/ine_census_2021/CensoPersonas_2021.tab.zst"

with open(LOCAL_ZST, "rb") as f:
    raw = zstd.decompress(f.read())

sample = pd.read_csv(io.BytesIO(raw), sep="\t", nrows=1000, dtype=str)
print(f"Columns ({len(sample.columns)}): {list(sample.columns)}\n")

for col in ["TENEN_VIV", "TIPO_EDIF_VIV", "CPRO", "CMUN"]:
    print(f"{col}: {sample[col].value_counts().to_dict()}")

Columns (129): ['CPRO', 'CMUN', 'NVIV', 'NORDEN', 'MNAC', 'ANAC', 'VAREDAD', 'SEXO', 'PNACIO', 'PNACIM', 'CPRO_NAC', 'CMUN_NAC', 'RESI_NACIM', 'VARANORES', 'VARANOM', 'ANOP', 'VARANOC', 'VARANOE', 'PAIS_PR', 'PROV_PR', 'CMUN_PR', 'PAIS_ANT', 'CPRO_ANT', 'CMUN_ANT', 'RESI_ANT', 'PAIS_UNANO', 'CPRO_UNANO', 'CMUN_UNANO', 'RESI_UNANO', 'PAIS_DANO', 'CPRO_DANO', 'CMUN_DANO', 'RESI_DANO', 'ECIVIL', 'ESREAL_CNEDA', 'ESCUR', 'ESCUR2', 'TESCUR', 'RELA', 'OCU63', 'ACT89', 'SITU', 'LTRAB', 'CPRO_TRAB', 'CMUN_TRAB', 'LEST', 'CPRO_EST', 'CMUN_EST', 'TIPO_EDIF_VIV', 'SUP_VIV', 'TENEN_VIV', 'SUP_OCU_VIV', 'NPLANTAS_SOBRE_EDIF', 'NPLANTAS_BAJO_EDIF', 'ANO_CONS', 'NORDEN_PAD', 'NORDEN_MAD', 'NORDEN_CON', 'NORDEN_OPA', 'FAMILIA', 'NUCLEO', 'TIPOPER', 'SEXO_MAD', 'VAREDAD_MAD', 'PNACIM_MAD_GR9', 'NACIO_MAD_GR10', 'ECIVL_MAD', 'ESREAL_MAD_GR5', 'RELA_MAD', 'SITU_MAD', 'SEXO_PAD', 'VAREDAD_PAD', 'PNACIM_PAD_GR9', 'NACIO_PAD_GR10', 'ECIVL_PAD', 'ESREAL_PAD_GR5', 'RELA_PAD', 'SITU_PAD', 'SEXO_CON', 'VAREDAD_

In [3]:
# CELL 3 — CONVERT TO PARQUET WITH COMMERCIAL FILTER AND COLUMN SELECTION
#
# Filter: TENEN_VIV IN ('2','3')
#   2 = En propiedad (owner-occupied)   ← install decision maker
#   3 = En alquiler (private rented)
#   Excluded: 4 = other (social/free housing), 9 = unknown
#
# Column selection: keep geography+linkage, person demographics, dwelling,
#   kinship structure, denormalized parent/spouse summaries, household/nucleus.
#   Drop: _MI imputation flags, mobility history, nationality detail, work/study location.

import duckdb, pathlib, os

LOCAL_ZST = "data/input/ine_census_2021/CensoPersonas_2021.tab.zst"
OUT       = "data/generated/ine_census_2021/censo_personas.parquet"
pathlib.Path("data/generated/ine_census_2021").mkdir(parents=True, exist_ok=True)

KEEP_COLS = [
    # Geography + household linkage
    "CPRO", "CMUN", "NVIV", "NORDEN",
    # Person demographics
    "ANAC", "VAREDAD", "SEXO", "ECIVIL",
    "ESREAL_CNEDA", "RELA", "OCU63", "ACT89", "SITU", "VARANORES",
    # Dwelling (attached to every person row)
    "TIPO_EDIF_VIV", "SUP_VIV", "TENEN_VIV", "NPLANTAS_SOBRE_EDIF", "ANO_CONS",
    # Kinship linkage — enables intergenerational modelling
    "NORDEN_PAD", "NORDEN_MAD", "NORDEN_CON", "NORDEN_OPA",
    "FAMILIA", "NUCLEO", "TIPOPER",
    # Denormalized parent 1 summary
    "VAREDAD_MAD", "ESREAL_MAD_GR5", "RELA_MAD", "SITU_MAD",
    # Denormalized parent 2 summary
    "VAREDAD_PAD", "ESREAL_PAD_GR5", "RELA_PAD", "SITU_PAD",
    # Denormalized spouse/partner summary
    "VAREDAD_CON", "ESREAL_CON_GR5", "RELA_CON", "SITU_CON",
    # Household
    "TAM_HOG", "ESTRUC_HOG", "TIPO_HOG",
    # Nucleus
    "TIPO_NUC", "TAM_NUC", "NHIJOS_NUC", "TIPO_PAR_NUC1",
]

col_list = ", ".join(KEEP_COLS)

con = duckdb.connect()
con.execute(f"""
    COPY (
        SELECT {col_list}
        FROM read_csv(
            '{LOCAL_ZST}',
            delim='\t', header=true, all_varchar=true, compression='zstd'
        )
        WHERE TENEN_VIV IN ('2', '3')
    )
    TO '{OUT}'
    (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
""")

size_mb = pathlib.Path(OUT).stat().st_size / 1_048_576
n_rows  = con.execute(f"SELECT COUNT(*) FROM '{OUT}'").fetchone()[0]
n_cols  = len(con.execute(f"SELECT * FROM '{OUT}' LIMIT 0").description)
print(f"Saved : {OUT}")
print(f"Shape : {n_rows:,} rows × {n_cols} columns  ({size_mb:.0f} MB)")
print(f"Filter: TENEN_VIV IN ('2','3') — owner-occupied + private rented only")

Saved : data/generated/ine_census_2021/censo_personas.parquet
Shape : 4,348,375 rows × 45 columns  (59 MB)
Filter: TENEN_VIV IN ('2','3') — owner-occupied + private rented only


In [4]:
# CELL 5 — UPLOAD PARQUET TO S3
import subprocess

S3_OUT = f"s3://{BUCKET}/ine-census-2021/generated/censo_personas.parquet"

result = subprocess.run(
    ["aws", "s3", "cp", OUT, S3_OUT, "--profile", PROFILE],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print(f"Uploaded: {S3_OUT}")
else:
    print("Upload failed:")
    print(result.stderr)

Uploaded: s3://hsf-group-ai-spain-hvac/ine-census-2021/generated/censo_personas.parquet
